<a href="https://colab.research.google.com/github/AdityaSinghDevs/AdityaSinghDevs/blob/main/learn/autograd.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Autograd

In [219]:
import torch

In [220]:
x = torch.tensor([12], dtype=torch.float32, requires_grad=True)
x

tensor([12.], requires_grad=True)

In [221]:
y  = x**2
y

tensor([144.], grad_fn=<PowBackward0>)

In [222]:
y.backward()
x.grad

tensor([24.])

# For more complex functions

In [223]:
# without autograd
import math

def dz_dx(x):
  return 2*x* math.cos(x**2)

dz_dx(3)

-5.466781571308061

In [224]:
# with autograd
x = torch.tensor(3.0 , requires_grad = True)

In [225]:
y = x**2

In [226]:
z = torch.sin(y)   # creating a basic trignometric fn

In [227]:
print(x)
print(y)
print(z)

tensor(3., requires_grad=True)
tensor(9., grad_fn=<PowBackward0>)
tensor(0.4121, grad_fn=<SinBackward0>)


In [228]:
z.backward()
x.grad

tensor(-5.4668)

# Now for a NN

## perceptron diagram in register, but simple NN

## values
| CGPA(x) | Placement(y)  |
|:--------|:--------------|
| 6.7     |   0.0         |


##Linear Transformation -> z = w.x + b
## activation -> y_pred  = sigmoid(z)
## Loss -> L = [ y . ln(y_pred) + (1 - y) . ln(1-y_pred) ]

## Therefore manually calculating grad of w wrt L
### - grad of L wrt y_pred -> dL/dy_pred  =  (y_pred - y)/y_pred(1-y_pred)
### - grad of y_pred wrt to z -> dy_pred/dz y_pred(1-y_pred)
### - grad of z wrt w ->  dz/dw = x and dz/db = 1

In [229]:
#Inputs
x = torch.tensor(6.7) # Input value
y = torch.tensor(0.0) #True label (binary)

w = torch.tensor(1.0 ) #wt
b = torch.tensor(0.0) #bias

In [230]:
# Binary Cross-Entropy Loss for scalar
def binary_cross_entropy_loss(prediction, target):
  epsilon = 1e-8  # Small value to avoid log(0)
  prediction = torch.clamp(prediction, epsilon, 1 - epsilon)
  return -(target * torch .log(prediction) + (1 - target) * torch.log(1-prediction))

In [231]:
# forward pass
z = w * x + b #wted sum
y_pred = torch.sigmoid(z) # predicted probability

#computing loss
loss = binary_cross_entropy_loss(y_pred, y)
loss

tensor(6.7012)

In [232]:
# Derivatives (Already calculated values)
#1. dL/dy_pred : Loss wrt prediction
dloss_dy_pred = (y_pred - y)/(y_pred*(1-y_pred))

#2 dy_pred/dz : Pred (y_pred) wrt z (sigmpoid fn)
dy_pred_dz = y_pred*(1-y_pred)

#3. dz/dw and dz/db : z wrt w and b
dz_dw = x # dz/dw = x
dz_db = 1 # dz/db = 1 (bias directly contributes to z)

dL_dw = dloss_dy_pred * dy_pred_dz * dz_dw
dL_db = dloss_dy_pred * dy_pred_dz * dz_db

In [233]:
print(f" Manual Grad of loss wrt w : {dL_dw}")
print(f" Manual Grad of loss wrt b : {dL_db}")

 Manual Grad of loss wrt w : 6.691762447357178
 Manual Grad of loss wrt b : 0.998770534992218


# USING AUTOGRAD NOW (MENTOS ZINDaGI)

In [234]:
x = torch.tensor(6.7) # No grad required as wont be computed for z or y
y = torch.tensor(0.0)

In [235]:
w = torch.tensor(1.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True) # grads required here hence true
print(w, b)

tensor(1., requires_grad=True) tensor(0., requires_grad=True)


In [236]:
z = w * x + b
z

tensor(6.7000, grad_fn=<AddBackward0>)

In [237]:
y_pred = torch.sigmoid(z)
y_pred

tensor(0.9988, grad_fn=<SigmoidBackward0>)

In [238]:
loss = binary_cross_entropy_loss(y_pred , y)
loss

tensor(6.7012, grad_fn=<NegBackward0>)

In [239]:
# Now using autograd backprop done easily
loss.backward()
w.grad
b.grad

tensor(0.9988)

# Uing autograd with vectors

In [240]:
x = torch.tensor([1.0,2.0,4.0,5.0] , requires_grad=True)

In [241]:
#FORWARD PASS
y = (x**2).mean()
y

tensor(11.5000, grad_fn=<MeanBackward0>)

In [242]:
# BACKWARD PASS
y.backward()
x.grad

tensor([0.5000, 1.0000, 2.0000, 2.5000])

# Gradients on Backward pass get accumulated whenever forward and backward pass are run again

## Therefore gradient clearance is done

In [243]:
x.grad.zero_()

tensor([0., 0., 0., 0.])

# Disable gradient tracking:
Say after training is completed only forward pass is required, hence you dont need backprop again and again to run at that time, hence grad track needs to be stopped

In [244]:
x = torch.tensor(1.0, requires_grad = True)
x

tensor(1., requires_grad=True)

In [245]:
y = x**2 # FORWARD
y

tensor(1., grad_fn=<PowBackward0>)

In [246]:
y.backward()
x.grad

In [249]:
# disabling
# option 1 - requires_grad(False)
# x.requires_grad(False)
# #now y.backward wont work

# option 2 - detach() : Creates a new tensor and dettaches it from the comp graph
z = x.detach()
z # will have same value of x

# y.backward()
# x.grad

# y1 = z **2
# y1.backward()
# z.grad #cant be calculated

# option 3 - torch.no_grad() : MOST CONVENIENT

with torch.no_grad():
  y2 = x ** 2

y2.backward # wont work

<bound method Tensor.backward of tensor(1.)>